#### Heat Equation Part C
# <span style="color:blue">Boundary conditions </span>
---

## Learning objectives

By the end of this part, you will be able to:

- Understand how boundary conditions (Dirichlet vs Neumann) modify the structure of the discrete Laplacian matrix.

- Relate changes in the system matrix to:

  - physical modeling assumptions,

  - qualitative solution behavior,

  - the spectrum (eigenvalues and eigenmodes) of the system.

- Construct and compare discrete Laplacian matrices corresponding to different boundary conditions using a matrix-based approach.

- Interpret how boundary conditions affect the spatial heat modes and long-time behavior of the solution.


## Contents
1. Boundary conditions as matrix structure
2. How boundary conditions enter the matrix formulation
3. Boundary conditions determine eigenmodes
4. Dirichlet vs Neumann Boundary Conditions: Physical Sketch
5. Modeling implication
6. Effect of Boundary Conditions on the System Matrix
7. Matrix construction code
8. Eigenvalue and eigenvector comparison plot
9. Numerical experiment: long-time behavior

👉 Boundary conditions define how the system interacts with its surroundings.


---

## 1. Boundary conditions as matrix structure

Boundary conditions are not merely side assumptions added to a partial differential equation. \
In matrix-based numerical methods, **boundary conditions directly determine the structure of the system matrix** and, consequently, the **qualitative behavior of the numerical solution**.

Throughout Parts A and B of this project, we have used **homogeneous Dirichlet boundary conditions**:

$$
u(0,t) = 0, \qquad u(L,t) = 0,
$$

which correspond physically to a rod whose ends are held at zero temperature.

---

## 2. How boundary conditions enter the matrix formulation

In the finite difference discretization of the one-dimensional heat equation,:

- only **interior grid points** are treated as unknowns,
- boundary values are assumed known and are therefore **excluded from the unknown vector**.

As a result:

- the temperature vector $\mathbf{u}(t)$ contains only interior values,
- the system matrix $A$ represents the second derivative operator **restricted to the interior of the domain**.

This choice implicitly enforces the boundary conditions and leads to the familiar tridiagonal Laplacian matrix.

$$
A_D = \frac{1}{\Delta x^2}
\begin{bmatrix}
-2 & 1  &        &        &   \\
1  & -2 & 1      &        &   \\
& \ddots & \ddots & \ddots &   \\
&        & 1      & -2 & 1 \\
&        &        &  1 & -2 \\
\end{bmatrix}.
$$

Thus, boundary conditions are not applied *after* the matrix system is formed; they are **built into the matrix itself** through the selection of unknowns and 
[<ins>**stencil structure**</ins>](https://en.wikipedia.org/wiki/Finite_difference_method#Explicit_method).


---

## 3. Boundary conditions determine eigenmodes

Because boundary conditions modify the system matrix $A$:

- they change the **eigenvalues** of $A$,
- they change the **eigenvectors** of $A$,
- and therefore they change the **spatial heat modes** of the system.

For example:

- **Dirichlet boundary conditions**
  - fix the **solution values** at the boundary,
  - lead to eigenvectors with sine-like shapes,
  - enforce zero values at the domain endpoints.
- **Neumann boundary conditions**
  - prescribe the **derivative** of the solution at the boundary,
  - lead to cosine-like eigenvectors,
  - allow nonzero boundary values and **introduce different long-time behavior**.

Thus, boundary conditions directly shape the spatial patterns through which heat diffuses by altering the **spectral structure** of the discrete Laplacian matrix.


#### Why this matters

From a matrix perspective, changing boundary conditions is not a minor modeling detail — it produces a **different linear dynamical system**. 

In the following sections, we will make this connection explicit by:

- constructing Laplacian matrices for different boundary conditions,

- comparing their eigenvalues and eigenvectors,

- and observing how these differences affect numerical solutions.

👉 **Different boundary choices lead to different matrix structures and solutions.**


---

## 4. Dirichlet vs Neumann Boundary Conditions: Physical Sketch

Before constructing the corresponding matrices, it is helpful to visualize the physical meaning of the two boundary conditions considered in this project.

### Dirichlet Boundary Conditions

Under **Dirichlet boundary conditions**, the temperature at the ends of the rod is fixed (here, zero):
$$
u(0,t) = 0, \quad u(L,t) = 0.
$$

**Physical interpretation**

* The ends of the rod are held at prescribed temperatures by external reservoirs.
* Heat can flow only through the boundaries, but the boundary values themselves do not change. The interior does not lose heat except by conduction toward these boundaries.


**Sketch description**

* The temperature profile touches the horizontal axis at both ends.
* All admissible temperature profiles vanish at the boundaries.
* Spatial modes resemble sine waves with nodes at the endpoints.

This constraint enforces zero values at the first and last grid points and leads to eigenmodes that are zero at the domain boundaries.



### Neumann Boundary Conditions

Under **Neumann boundary conditions**, the heat flux at the ends of the rod is prescribed:
$$
\frac{\partial u}{\partial x}(0,t) = 0, \quad \frac{\partial u}{\partial x}(L,t) = 0.
$$

**Physical interpretation**

* The rod is thermally insulated at both ends.
* No heat flows across the boundaries. The heat just spreads inside the rod.

**Sketch description**

* The temperature profile has zero slope at the boundaries.
* The temperature profile has horizontal tangents at the boundaries; it does not need to cross or touch any particular value there. 
* Spatial modes resemble cosine waves, including a constant mode.

This constraint modifies the finite difference stencil at the boundaries and produces a system matrix with different spectral properties, including a zero eigenvalue corresponding to steady-state solutions.

---

## 5. Modeling implication

Although both boundary conditions describe diffusion on the same spatial domain, they impose fundamentally different constraints on admissible solutions. These constraints are reflected directly in:

* the structure of the discrete Laplacian matrix,
* the shape of the eigenmodes,
* and the long-time behavior of the heat equation.

In the next section, we make these distinctions precise by constructing and comparing the corresponding system matrices.

✅ The mathematical formulation must consistently incorporate boundary information.


---

## 6. Effect of Boundary Conditions on the System Matrix

The influence of boundary conditions becomes explicit when examining the **first and last rows** of the discrete Laplacian matrix. These rows encode how the finite difference stencil is modified at the domain boundaries.

Let the spatial grid consist of $ N $ interior points with spacing $ \Delta x $, and let
$ \mathbf{u}(t) \in \mathbb{R}^N $ denote the vector of interior temperatures.



### Dirichlet Boundary Conditions

For homogeneous Dirichlet boundary conditions,
$$
u(0,t) = 0, \quad u(L,t) = 0,
$$
the boundary values are known and excluded from the unknown vector.

The discrete Laplacian matrix has the form

$$
A_D = \frac{1}{\Delta x^2}
\begin{bmatrix}
\textcolor{blue}{-2} & 1  &        &        &   \\
1  & -2 & 1      &        &   \\
& \ddots & \ddots & \ddots &   \\
&        & 1      & -2 & 1 \\
&        &        &  1 & \textcolor{blue}{-2} \\
\end{bmatrix}.
$$

**First row**
$$
\frac{1}{\Delta x^2}[\textcolor{blue}{-2} \ \ 1  \ \ 0 \ \ \cdots \ \ 0]
$$

**Last row**
$$
\frac{1}{\Delta x^2}[0 \ \ \cdots \ \ 0 \ \ 1  \ \textcolor{blue}{-2}]
$$

The stencil assumes a fixed zero value outside the domain, which enforces the boundary conditions implicitly through the matrix structure.



### Neumann Boundary Conditions

For homogeneous Neumann boundary conditions,
$$
\frac{\partial u}{\partial x}(0,t) = 0, \quad \frac{\partial u}{\partial x}(L,t) = 0,
$$
the **derivative constraint** is enforced by modifying the finite difference approximation at the boundaries.

Here we use ghost points — an extra, fictitious grid point placed just outside the physical domain so that boundary conditions can be enforced cleanly. It is not a real point in the domain; only a mathematical trick. The resulting discrete Laplacian matrix then becomes





$$
A_N = \frac{1}{\Delta x^2}
\begin{bmatrix}
\textcolor{red}{-1} &  1  &        &        &   \\
 1  & -2 & 1      &        &   \\
& \ddots & \ddots & \ddots &   \\
&        & 1      & -2 & 1 \\
&        &        & 1  & \textcolor{red}{-1} \\
\end{bmatrix}.
$$

**First row**
$$
\frac{1}{\Delta x^2}[\textcolor{red}{-1} \ \ 1 \ \ 0 \ \ \cdots \ \ 0]
$$

**Last row**
$$
\frac{1}{\Delta x^2}[0 \ \ \cdots \ \ 0 \ \ 1 \ \ \textcolor{red}{-1}]
$$

Compared to the Dirichlet case, the diagonal entries at the boundaries are less negative, reflecting the absence of heat flux through the endpoints.



### Key Structural Differences

* Dirichlet boundary conditions enforce fixed endpoint values by effectively coupling the interior nodes to zero-valued exterior points.
* Neumann boundary conditions alter the boundary stencil to enforce zero derivative, changing only the **first and last rows** of the matrix.
* These local row modifications lead to global changes in:

  * the eigenvalues,
  * the eigenvectors,
  * and the long-time behavior of the system.

In particular, the **Neumann** matrix admits a **zero eigenvalue** corresponding to a **constant steady-state mode**, whereas the Dirichlet matrix does not.

👉 The system behavior is strongly influenced by how edges are treated.




**Ghost-point argument for Neumann conditions.**
The zero-flux condition $ \partial u/\partial x = 0 $ at the boundary is approximated by
$$
\frac{u_1 - u_{-1}}{2\Delta x} = 0 \ \ \Rightarrow \ \ u_{-1} = u_1,
$$
which, when substituted into the second-derivative stencil at the first grid point, yields
$$
\frac{u_1 - 2u_0 + u_{-1}}{\Delta x^2}
= \frac{2(u_1 - u_0)}{\Delta x^2},
$$
leading to the modified boundary row $ \frac{1}{\Delta x^2}[\textcolor{red}{-1} \ \ 1] $.

(Note:  The factor 2 in $ \frac{2(u_1 - u_0)}{\Delta x^2} $  is not “lost” above  — it is absorbed into the normalization of the boundary stencil so that the Neumann boundary row has the same overall scaling as the interior Laplacian. Writing the row as 
$ \frac{1}{\Delta x^2}[\textcolor{red}{-1} \ \ 1] $
 preserves symmetry, consistency, and the correct operator structure without artificially doubling the boundary coefficients.)




**Dirichlet contrast (one sentence).**
For homogeneous Dirichlet conditions $ u(0,t)=0 $, the ghost value is fixed as $ u_{-1}=0 $, so substituting into the second-derivative stencil yields
$$
\frac{u_1 - 2u_0 + u_{-1}}{\Delta x^2}
= \frac{u_1 - 2u_0}{\Delta x^2},
$$
which produces the boundary row $$\frac{1}{\Delta x^2}[-2 \ \ 1]$$.






👉 **Boundary conditions** affect the discrete heat equation only through the first and last rows of the system matrix, but these local modifications alter the entire spectrum, changing spatial modes, decay rates, and long-time behavior of the solution.

👉 **Dirichlet** fixes values, **Neumann** fixes flux; → matrix rows change accordingly.

---

## 7. Matrix construction code

✅ Boundary conditions are not just constraints—they shape the entire solution.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rc
rc('animation', html='jshtml')
from matplotlib.animation import FuncAnimation


In [ ]:
def laplacian_dirichlet(N, dx):
    """
    Discrete 1D Laplacian with homogeneous Dirichlet boundary conditions.
    Unknowns correspond to N interior grid points.
    """
    A = np.zeros((N, N))
    
    for i in range(N):
        A[i, i] = -2
        if i > 0:
            A[i, i-1] = 1
        if i < N-1:
            A[i, i+1] = 1
            
    return A / dx**2


def laplacian_neumann(N, dx):
    """
    Discrete 1D Laplacian with homogeneous Neumann boundary conditions.
    Unknowns correspond to N grid points.
    """
    A = np.zeros((N, N))
    
    # interior points
    for i in range(1, N-1):
        A[i, i]   = -2
        A[i, i-1] = 1
        A[i, i+1] = 1

    # boundary rows
    A[0, 0]     = -1
    A[0, 1]     =  1
    A[-1, -2]   =  1
    A[-1, -1]   = -1
    
    return A / dx**2


# Example usage
N = 6                             
L = 1.0
dx = L / (N + 1)

A_D = laplacian_dirichlet(N, dx)
A_N = laplacian_neumann(N, dx)

print("Dirichlet Laplacian matrix:\n")
print(A_D)

print("\nNeumann Laplacian matrix:\n")
print(A_N)

## 8. Eigenvalue and eigenvector comparison plot

To make the effect of boundary conditions concrete, we compare the eigenvalues and eigenvectors of the Dirichlet and Neumann discrete Laplacian matrices. Although the matrices differ only in their boundary rows, their spectral properties — and therefore their spatial heat modes — are qualitatively different.

In [ ]:
# Build matrices for eigenvals/eigenvecs plotting  

N = 40                             
L = 1.0
dx = L / (N + 1)
A_D = laplacian_dirichlet(N, dx)
A_N = laplacian_neumann(N, dx)


In [ ]:
# Compute eigenvalues/eigenvectors
eigvals_D, eigvecs_D = np.linalg.eigh(A_D)
eigvals_N, eigvecs_N = np.linalg.eigh(A_N)

# Sort in descending order (largest eigenvalue first)
# This makes physical ordering consistent:
# Neumann: mode 0 (λ=0) becomes index 0
# Dirichlet: lowest frequency becomes index 0

# Indexes sort from end to beginning: 29,...,0
idx_D = np.argsort(eigvals_D)[::-1]
idx_N = np.argsort(eigvals_N)[::-1]

# Eigenvalues from largest to smallest
eigvals_D = eigvals_D[idx_D]            
eigvecs_D = eigvecs_D[:, idx_D]

eigvals_N = eigvals_N[idx_N]
eigvecs_N = eigvecs_N[:, idx_N]


# Fix eigenvector sign ambiguity 
def fix_sign(eigvecs):
    for i in range(eigvecs.shape[1]):
        if eigvecs[1, i] < 0:
            eigvecs[:, i] *= -1
    return eigvecs

eigvecs_D = fix_sign(eigvecs_D)
eigvecs_N = fix_sign(eigvecs_N)

#  Now:
#  - eigvals_N[0] → λ₀ = 0 (constant mode)
#  - eigvals_N[1] → first cosine mode
#  - eigvals_D[0] → first sine mode

# Analytical eigenfunctions for comparison (continuous)
def analytical_dirichlet(x, k, L=1.0):
    # k = 1,2,3,...
    return np.sin(k * np.pi * x / L)

def analytical_neumann(x, k, L=1.0):
    # k = 0,1,2,3,...
    return np.cos(k * np.pi * x / L)



In [ ]:
# Plotting

modes_to_plot = 3

x_D = np.linspace(0, 1, N+2)
x_N = np.linspace(0, 1, N+2)

plt.figure(figsize=(12, 8))

for k in range(modes_to_plot):

    # ------- DIRICHLET --------- #
    
    mode_D = np.zeros(N+2)
    mode_D[1:-1] = eigvecs_D[:, k]

    ana_D = analytical_dirichlet(x_D, k+1)

    # normalize analytical to numerical amplitude
    ana_D = ana_D / np.max(np.abs(ana_D)) * np.max(np.abs(mode_D))

    # Align sign with analytical
    if np.dot(mode_D, ana_D) < 0:
        mode_D *= -1

    plt.subplot(2, modes_to_plot, k+1)
    plt.plot(x_D, mode_D, ".", label="Numerical")
    plt.plot(x_D, ana_D, linewidth=1, label="Analytical")
    plt.title(f"Dirichlet mode {k+1}\nλ={eigvals_D[k]:.2f}")
    plt.grid(True)

    if k == 0:
        plt.legend()

    # --------  NEUMANN  -------- #
    
    mode_index = k + 1                 # k+1 because index 0 is λ0 constant mode and plotted separately

    mode_N = np.zeros(N+2)
    mode_N[1:-1] = eigvecs_N[:, mode_index]

    # enforce zero derivative at boundaries
    mode_N[0]  = mode_N[1]
    mode_N[-1] = mode_N[-2]

    ana_N = analytical_neumann(x_N, k+1)
    
    # normalize analytical to numerical amplitude
    ana_N = ana_N / np.max(np.abs(ana_N)) * np.max(np.abs(mode_N))

    # Align sign with analytical
    if np.dot(mode_N, ana_N) < 0:
        mode_N *= -1

    plt.subplot(2, modes_to_plot, modes_to_plot + k + 1)
    plt.plot(x_N, mode_N, ".", label="Numerical")
    plt.plot(x_N, ana_N,  linewidth=1, label="Analytical")
    plt.title(f"Neumann mode {k+1}\nλ={eigvals_N[mode_index]:.2f}")
    plt.grid(True)

    if k == 0:
        plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
# Separate figure: Neumann mode 0 (λ = 0)

plt.figure(figsize=(4, 4))  # same size as panel figure

# mode 0 is index 0 after descending sort
mode0 = np.zeros(N+2)
mode0[1:-1] = eigvecs_N[:, 0]

# enforce zero derivative at boundaries
mode0[0]  = mode0[1]
mode0[-1] = mode0[-2]

# analytical constant mode
ana0 = analytical_neumann(x_N, 0)

# normalize analytical to numerical amplitude
ana0 = ana0 / np.max(np.abs(ana0)) * np.max(np.abs(mode0))

# align sign (just for consistency)
if np.dot(mode0, ana0) < 0:
    mode0 *= -1

plt.plot(x_N, mode0, ".", label="Numerical")
plt.plot(x_N, ana0, linewidth=1, label="Analytical")

plt.title(f"Neumann mode 0 (constant mode)\nλ={eigvals_N[0]:.2f}")
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()

#### Interpretation 

- Dirichlet eigenmodes vanish at the boundaries and resemble sine waves.

- Neumann eigenmodes have zero slope at the boundaries and resemble cosine waves.

- The Neumann system includes a constant eigenvector corresponding to a zero eigenvalue.

This visually confirms that boundary conditions determine admissible spatial heat modes.

---

## 9. Numerical experiment: long-time behavior

How a combination of two eigenmodes evolves in time?

(The time scale is intentionally compressed to emphasize the rapid decay of high-frequency components.)

In [ ]:
# --- Animation: evolution of the combination  of two eigenmodes --- #
#  Wait a minute, it will take a while to start.

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

# parameters
N = 100
L = 1.0
x = np.linspace(0, L, N)

# eigenmodes (sine basis for Dirichlet BCs)
mode1 = np.sin(np.pi * x)          # low frequency
mode2 = np.sin(5 * np.pi * x)      # higher frequency

# initial condition: combination
u0 = mode1 + 0.5 * mode2

# eigenvalues (continuous Laplacian)
lambda1 = (np.pi)**2
lambda2 = (5 * np.pi)**2

# time evolution (heat equation: exponential decay)
def solution(t):
    return (
        np.exp(-lambda1 * t) * mode1 +
        0.5 * np.exp(-lambda2 * t) * mode2
    )

# setup plot
fig, ax = plt.subplots()
line, = ax.plot(x, u0)
ax.set_ylim(-1.5, 1.5)
ax.set_title("Evolution of two eigenmodes")

frames = 120
T_max = 0.4

# animation update
def update(frame):
    t = (frame / frames)**2 * T_max
    line.set_ydata(solution(t))
    ax.set_title(f"t = {t:.3f}")
    return line,

from IPython.display import HTML

anim = FuncAnimation(fig, update, frames=200, interval=100)

plt.close(fig)  # <-- prevents duplicate static image

HTML(anim.to_jshtml())


## Evolution of eigenmodes

👉 **The solution of the heat equation** can be expressed as a **combination of eigenmodes of the discrete Laplacian**.

**Each eigenmode evolves independently** in time according to an exponential decay:
$$
u_k(t) \sim e^{-\lambda_k t}
$$

👉 In this example, the initial condition is a combination of:
- a low-frequency mode
- a higher-frequency mode

**Because the higher-frequency mode has a larger eigenvalue, it decays faster**.

✅ This means that, over time, **the solution becomes smoother and is dominated by the lowest-frequency components**.

This animation illustrates a key principle:

👉 The heat equation acts as a **frequency filter**, damping out high-frequency components more rapidly than low-frequency ones.

*Heikki Miettinen 2026*